In [ ]:
import logging
from kiteconnect import KiteConnect
import pandas as pd
import datetime as dt
import pandas as pd

In [41]:
request_token=open("request_token.txt","r").read()
key_secrets=open("api_key.txt","r").read().split()
kite = KiteConnect(api_key=key_secrets[0])
data = kite.generate_session(request_token=request_token,api_secret=key_secrets[1])

In [42]:
# Get dump of all NSE instruments
instruments_dump=kite.instruments("NSE")
instruments_df=pd.DataFrame(instruments_dump)
instruments_df.to_csv("NSE_Instruments.csv",index=False)

In [43]:
def instrumentLookup(instrument_df,symbol):
    """Looks up instrument token for a given script from instrument dump"""
    try:
        return instrument_df[instrument_df.tradingsymbol==symbol].instrument_token.values[0]
    except:
        return -1


In [44]:
def fetchOHLC(ticker,interval,duration):
    """extracts historical data and outputs in the form of dataframe"""
    instrument = instrumentLookup(instruments_df,ticker)
    data = pd.DataFrame(kite.historical_data(instrument,dt.date.today()-dt.timedelta(duration), dt.date.today(),interval))
    data.set_index("date",inplace=True)
    return data

In [46]:
ohlc = fetchOHLC("ASIANPAINT","5minute",30)
ohlc

,open,high,low,close,volume
date,,,,,
2025-03-21 09:15:00+05:30,2291.65,2291.65,2278.45,2283.50,14804
2025-03-21 09:20:00+05:30,2283.10,2283.85,2277.85,2278.05,9578
2025-03-21 09:25:00+05:30,2278.15,2284.00,2278.15,2282.05,9619
2025-03-21 09:30:00+05:30,2282.65,2284.00,2279.55,2282.00,13174
2025-03-21 09:35:00+05:30,2282.40,2285.00,2281.10,2284.10,7125
...,...,...,...,...,...
2025-04-17 15:05:00+05:30,2465.10,2471.10,2465.00,2470.70,70724
2025-04-17 15:10:00+05:30,2470.70,2470.90,2469.00,2469.40,14947
2025-04-17 15:15:00+05:30,2469.20,2469.40,2464.90,2465.00,15791


In [47]:
import pandas as pd
import datetime as dt

def fetchOHLCExtended(ticker, inception_date, interval):
    """Extracts historical data and outputs in the form of a DataFrame
       inception date string format - dd-mm-yyyy"""
    instrument = instrumentLookup(instruments_df, ticker)
    from_date = dt.datetime.strptime(inception_date, '%d-%m-%Y')
    to_date = dt.date.today()
    data = pd.DataFrame(columns=['date', 'open', 'high', 'low', 'close', 'volume'])

    while True:
        if from_date.date() >= (dt.date.today() - dt.timedelta(100)):
            historical_data = pd.DataFrame(kite.historical_data(instrument, from_date, dt.date.today(), interval))
            data = pd.concat([data, historical_data], ignore_index=True)
            break
        else:
            to_date = from_date + dt.timedelta(100)
            historical_data = pd.DataFrame(kite.historical_data(instrument, from_date, to_date, interval))
            data = pd.concat([data, historical_data], ignore_index=True)
            from_date = to_date

    data.set_index("date", inplace=True)
    return data

# Example usage
fetchOHLCExtended("INFY", "01-01-2024", "5minute")


,open,high,low,close,volume
date,,,,,
2024-01-01 09:15:00+05:30,1539.00,1542.90,1535.25,1538.25,76413
2024-01-01 09:20:00+05:30,1538.15,1539.50,1537.10,1538.60,36570
2024-01-01 09:25:00+05:30,1538.65,1542.55,1538.55,1540.60,33452
2024-01-01 09:30:00+05:30,1541.00,1541.20,1539.50,1541.05,44560
2024-01-01 09:35:00+05:30,1540.75,1542.00,1540.00,1540.00,16008
...,...,...,...,...,...
2025-04-17 15:05:00+05:30,1414.60,1418.50,1414.50,1418.50,245743
2025-04-17 15:10:00+05:30,1418.50,1422.10,1417.80,1418.70,321453
2025-04-17 15:15:00+05:30,1418.60,1419.30,1417.20,1419.10,230685
